# Week 11 Contest: Round 2

**Open this notebook when your instructor tells you to.**

The factory has had a busy few weeks. You now have two new files:

- `round2_train.csv.gz`: **newly labelled** rows from the period right after your original training data
- `test_final.csv.gz`: the rows you'll be scored on from now on. They come after `round2_train` in time.

Everything else about the contest is unchanged: same columns, same metric, 5 submissions, public leaderboard on
30% and private final score on 70%.

Your Round 1 model will probably not do as well as it did. Your job in this round is to work out **why** and fix
it. Open the data and compare it with the old data. Answer these before touching a model:

1. Do the columns in `round2_train` look like the columns in `train`? Which ones are different?
2. Is the failure rate the same?
3. Are there new kinds of rows? (Think about Week 10.)
4. How should you validate now? What should your validation set look like?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report

GROUP_NAME = "CHANGE-ME"
DATA_DIR = "contest_data"

train = pd.read_csv(f"{DATA_DIR}/train.csv.gz")
round2 = pd.read_csv(f"{DATA_DIR}/round2_train.csv.gz")
test_final = pd.read_csv(f"{DATA_DIR}/test_final.csv.gz")
features = [c for c in train.columns if c.startswith("f")]

print("train:", train.shape, "  round2_train:", round2.shape, "  test_final:", test_final.shape)
print("failure rate  train: %.3f   round2: %.3f" % (train["label"].mean(), round2["label"].mean()))

In [ ]:
import os, re
from datetime import datetime

def make_submission(test_df, preds, group, round_name, out_dir="my_submissions"):
    """Validate predictions and write  <group>__<round>__<HHMMSS>.csv  ready to hand in."""
    preds = np.asarray(preds).astype(int)
    assert len(preds) == len(test_df), f"Expected {len(test_df)} predictions, got {len(preds)}"
    assert set(np.unique(preds)) <= {0, 1}, "Predictions must be hard 0/1 labels, not probabilities"
    group = re.sub(r"[^A-Za-z0-9]+", "-", group).strip("-")
    os.makedirs(out_dir, exist_ok=True)
    path = f"{out_dir}/{group}__{round_name}__{datetime.now():%H%M%S}.csv"
    pd.DataFrame({"id": test_df["id"].values, "pred": preds}).to_csv(path, index=False)
    print(f"Wrote {path}  ({preds.mean():.1%} predicted positive)")
    return path

## Compare the two periods

Start here, then keep going on your own.

In [ ]:
comp = pd.DataFrame({
    "train_mean": train[features].mean(),
    "round2_mean": round2[features].mean(),
    "train_std": train[features].std(),
    "round2_std": round2[features].std(),
})
comp["mean_shift_in_std"] = (comp["round2_mean"] - comp["train_mean"]) / comp["train_std"]
comp.reindex(comp["mean_shift_in_std"].abs().sort_values(ascending=False).index).head(10).round(3)

## Your turn

Build your best model for `test_final`, then submit with:

```python
make_submission(test_final, preds, GROUP_NAME, "final")
```

## Before the debrief, write down (a few sentences each)

1. What exactly was different in Round 2 and how did you find it?
2. What did your Round 1 model get wrong because of it?
3. What did you change, and what did that change to your validation score?
4. What did you try that **didn't** help?
5. If this were a real factory, what would you tell the operations manager to check?